# Bird Song Generation
## Stage 3: Classifier Architecture Study

This notebook is the visual, runnable companion to the Stage 3 classifier code. It compares four spectrogram classifiers for American Robin, Northern Cardinal, and Song Sparrow: a residual CNN, a plain CNN, a depthwise CNN, and a CNN-GRU. All training and test-evaluation switches default to `False`.

## Goal

1. Visualize the shared log-mel input representation.
2. Compare model capacity and architectural assumptions.
3. Train every architecture with identical splits, preprocessing, hyperparameters, and seed sets.
4. Select an architecture using validation results only.
5. Evaluate the selected checkpoint once on the held-out test split.

## Setup

The notebook imports model definitions and checkpoint logic from `src/` instead of maintaining a second implementation. This keeps notebook experiments consistent with the command-line pipeline.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

# Work when opened from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
SOURCE_ROOT = PROJECT_ROOT / 'src'
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from bird_song.classifier.model import ARCHITECTURES, build_classifier, count_trainable_parameters
from bird_song.config import SpectrogramConfig
from bird_song.data import ManifestDataset, make_loader, resolve_dataset_root
from bird_song.runtime import load_checkpoint, seed_everything

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.set_float32_matmul_precision('high')
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Project root:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

### Experiment Configuration

The default comparison repeats each architecture with three seeds. `NOTEBOOK_WORKERS = 0` is safe for interactive data previews; the standalone training subprocess can use more workers on Windows.

In [ ]:
DATASET_ROOT = PROJECT_ROOT / 'bird_songs_dataset'
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'spectrogram.json'
TRAIN_MANIFEST = PROJECT_ROOT / 'manifests' / 'full_dataset_train.csv'
VALIDATION_MANIFEST = PROJECT_ROOT / 'manifests' / 'full_dataset_validation.csv'
TEST_MANIFEST = PROJECT_ROOT / 'manifests' / 'full_dataset_test.csv'
REFERENCE_CHECKPOINT = PROJECT_ROOT / 'classifier_artifacts' / 'selected_crnn' / 'best.pt'
LEGACY_RESIDUAL_CHECKPOINT = PROJECT_ROOT / 'classifier_artifacts' / 'Harvey_classifier' / 'best.pt'
SINGLE_RUN_ROOT = PROJECT_ROOT / 'runs' / 'notebook_classifier'
COMPARISON_ROOT = PROJECT_ROOT / 'runs' / 'classifier_architectures'

SELECTED_ARCHITECTURE = 'crnn'
ARCHITECTURES_TO_COMPARE = list(ARCHITECTURES)
COMPARISON_SEEDS = [42, 123, 777]
SEED = 42
BATCH_SIZE = 64
NOTEBOOK_WORKERS = 0
TRAINING_WORKERS = 4
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
WIDTH = 32
DROPOUT = 0.30

RUN_SINGLE_MODEL = False
RUN_ARCHITECTURE_COMPARISON = False
OVERWRITE_RUNS = False
RUN_TEST_EVALUATION = False
FINAL_CHECKPOINT = REFERENCE_CHECKPOINT

assert SELECTED_ARCHITECTURE in ARCHITECTURES
assert set(ARCHITECTURES_TO_COMPARE).issubset(ARCHITECTURES)
seed_everything(SEED)

## Data

All models receive the same normalized 128 by 128 log-mel representation. Training uses waveform and SpecAugment transformations; validation and test preprocessing are deterministic. Splits are recording-safe, so segments from one source recording cannot cross splits.

In [ ]:
dataset_root = resolve_dataset_root(PROJECT_ROOT, DATASET_ROOT)
spectrogram_config = SpectrogramConfig.from_json(CONFIG_PATH)
classes = tuple(sorted(pd.read_csv(TRAIN_MANIFEST)['name'].unique()))

train_set = ManifestDataset(TRAIN_MANIFEST, dataset_root, classes, spectrogram_config, training=True)
validation_set = ManifestDataset(VALIDATION_MANIFEST, dataset_root, classes, spectrogram_config, training=False)
test_set = ManifestDataset(TEST_MANIFEST, dataset_root, classes, spectrogram_config, training=False)

train_loader = make_loader(
    train_set, BATCH_SIZE, NOTEBOOK_WORKERS, training=True, seed=SEED
)
validation_loader = make_loader(validation_set, BATCH_SIZE, NOTEBOOK_WORKERS)
test_loader = make_loader(test_set, BATCH_SIZE, NOTEBOOK_WORKERS)

split_summary = pd.concat([
    pd.read_csv(TRAIN_MANIFEST).assign(split='train'),
    pd.read_csv(VALIDATION_MANIFEST).assign(split='validation'),
    pd.read_csv(TEST_MANIFEST).assign(split='test'),
])[['split', 'name']].value_counts().rename('clips').reset_index()
display(split_summary)
print('Classes:', classes)
print('Input shape:', (1, spectrogram_config.n_mels, spectrogram_config.spectrogram_width))

### Visualize a Training Batch

The examples can include random crop, gain, noise, frequency masking, or time masking because they come from the training loader.

In [ ]:
specs, labels, paths = next(iter(train_loader))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
for axis, spec, label in zip(axes, specs[:3], labels[:3]):
    image = axis.imshow(
        spec.squeeze(0), origin='lower', aspect='auto', cmap='magma', vmin=-1, vmax=1
    )
    axis.set_title(classes[int(label)])
    axis.set_xlabel('Time frame')
axes[0].set_ylabel('Mel bin')
fig.colorbar(image, ax=axes, label='Normalized log-mel')
plt.show()
print('Batch shape:', tuple(specs.shape))
print('First file:', paths[0])

## Architecture Hypotheses

| Architecture | Hypothesis | Structural difference |
|---|---|---|
| `residual_cnn` | Skip connections make a deeper CNN easier to optimize. | Six residual blocks and dual global pooling |
| `plain_cnn` | A conventional convolution stack may be sufficient for three species. | No residual paths |
| `depthwise_cnn` | Similar spatial processing may work with far fewer parameters. | Depthwise-separable convolutions |
| `crnn` | Explicit temporal sequence modeling may help distinguish song patterns. | CNN features followed by a bidirectional GRU |

The architectures deliberately have different parameter counts. The experiment reports capacity alongside accuracy rather than claiming a parameter-matched ablation.

### Compare Model Capacity

Every model is constructed through the same factory used by the training script. The logarithmic axis keeps the compact depthwise model visible beside the larger CNNs.

In [ ]:
architecture_labels = {
    'residual_cnn': 'Residual CNN',
    'plain_cnn': 'Plain CNN',
    'depthwise_cnn': 'Depthwise CNN',
    'crnn': 'CNN-GRU',
}
parameter_rows = []
for architecture in ARCHITECTURES_TO_COMPARE:
    candidate = build_classifier(architecture, len(classes), DROPOUT, WIDTH)
    parameter_rows.append({
        'architecture': architecture,
        'label': architecture_labels[architecture],
        'trainable_parameters': count_trainable_parameters(candidate),
    })
parameter_table = pd.DataFrame(parameter_rows).sort_values('trainable_parameters')
display(parameter_table[['label', 'trainable_parameters']].style.format({'trainable_parameters': '{:,}'}))

fig, axis = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
bars = axis.barh(parameter_table['label'], parameter_table['trainable_parameters'], color='steelblue')
axis.set_xscale('log')
axis.set_xlabel('Trainable parameters (log scale)')
axis.set_title('Classifier capacity by architecture')
for bar, value in zip(bars, parameter_table['trainable_parameters']):
    axis.text(value * 1.04, bar.get_y() + bar.get_height() / 2, f'{value:,}', va='center')
plt.show()

### Inspect the Selected Architecture

Change `SELECTED_ARCHITECTURE` in the configuration cell to inspect another model. This cell performs only a small forward-shape check when you execute it; it does not train.

In [ ]:
selected_model = build_classifier(
    SELECTED_ARCHITECTURE, num_classes=len(classes), dropout=DROPOUT, width=WIDTH
).to(device)
with torch.inference_mode():
    example_logits = selected_model(specs[:2].to(device))
print(selected_model)
print(f'Architecture: {SELECTED_ARCHITECTURE}')
print(f'Trainable parameters: {count_trainable_parameters(selected_model):,}')
print('Logit shape:', tuple(example_logits.shape))
del selected_model

## Controlled Training Protocol

All architecture runs use the same manifests, augmentation, width, dropout, AdamW optimizer, learning rate, weight decay, label smoothing, cosine schedule, gradient clipping, maximum epochs, and early-stopping rule. Repeating seeds measures sensitivity to initialization and stochastic training. The held-out test manifest is excluded from architecture selection.

In [ ]:
def run_project_script(script_name, arguments):
    environment = os.environ.copy()
    existing_pythonpath = environment.get('PYTHONPATH')
    environment['PYTHONPATH'] = str(SOURCE_ROOT) + (
        os.pathsep + existing_pythonpath if existing_pythonpath else ''
    )
    command = [sys.executable, str(PROJECT_ROOT / 'scripts' / script_name), *map(str, arguments)]
    print('Running:', ' '.join(command))
    return subprocess.run(command, cwd=PROJECT_ROOT, env=environment, check=True)

common_training_arguments = [
    '--dataset-root', DATASET_ROOT,
    '--spectrogram-config', CONFIG_PATH,
    '--train-manifest', TRAIN_MANIFEST,
    '--val-manifest', VALIDATION_MANIFEST,
    '--epochs', EPOCHS,
    '--patience', PATIENCE,
    '--batch-size', BATCH_SIZE,
    '--workers', TRAINING_WORKERS,
    '--learning-rate', LEARNING_RATE,
    '--weight-decay', WEIGHT_DECAY,
    '--label-smoothing', LABEL_SMOOTHING,
    '--width', WIDTH,
    '--dropout', DROPOUT,
    '--device', device,
]

### Optional: Train One Architecture

Set `RUN_SINGLE_MODEL = True` to train only `SELECTED_ARCHITECTURE`. Existing outputs are protected unless `OVERWRITE_RUNS = True`.

In [ ]:
single_run_dir = SINGLE_RUN_ROOT / SELECTED_ARCHITECTURE / f'seed_{SEED}'
if RUN_SINGLE_MODEL:
    arguments = [
        '--architecture', SELECTED_ARCHITECTURE,
        '--seed', SEED,
        '--output-dir', single_run_dir,
        *common_training_arguments,
    ]
    if OVERWRITE_RUNS:
        arguments.append('--overwrite')
    run_project_script('03_train_classifier.py', arguments)
else:
    print('Skipped single-model training. Set RUN_SINGLE_MODEL = True to enable it.')

### Optional: Run the Full Architecture Comparison

Set `RUN_ARCHITECTURE_COMPARISON = True` to train every selected architecture for every seed. With the defaults, this launches 12 independent runs and writes a reproducible protocol plus CSV and Markdown summaries.

In [ ]:
if RUN_ARCHITECTURE_COMPARISON:
    arguments = [
        '--architectures', *ARCHITECTURES_TO_COMPARE,
        '--seeds', *COMPARISON_SEEDS,
        '--output-dir', COMPARISON_ROOT,
        *common_training_arguments,
    ]
    if OVERWRITE_RUNS:
        arguments.append('--overwrite')
    run_project_script('03_compare_classifier_architectures.py', arguments)
else:
    print('Skipped architecture comparison. Set RUN_ARCHITECTURE_COMPARISON = True to enable it.')

## Validation Results

These cells visualize completed runs from `COMPARISON_ROOT`. They show a clear instruction instead of inventing results when the comparison has not been run. Error bars are sample standard deviations across seeds.

In [ ]:
summary_path = COMPARISON_ROOT / 'summary.csv'
runs_path = COMPARISON_ROOT / 'runs.csv'

if summary_path.is_file() and runs_path.is_file():
    architecture_summary = pd.read_csv(summary_path)
    architecture_runs = pd.read_csv(runs_path)
    display_columns = [
        'architecture', 'runs', 'trainable_parameters',
        'validation_accuracy_mean', 'validation_accuracy_std',
        'validation_macro_f1_mean', 'validation_macro_f1_std',
        'best_epoch_mean',
    ]
    display(architecture_summary[display_columns].style.format({
        'trainable_parameters': '{:,.0f}',
        'validation_accuracy_mean': '{:.2%}',
        'validation_accuracy_std': '{:.2%}',
        'validation_macro_f1_mean': '{:.2%}',
        'validation_macro_f1_std': '{:.2%}',
        'best_epoch_mean': '{:.1f}',
    }))
else:
    architecture_summary = None
    architecture_runs = None
    print('No completed comparison found at:', COMPARISON_ROOT)
    print('Enable RUN_ARCHITECTURE_COMPARISON, or copy completed comparison outputs to this directory.')

### Accuracy, Macro F1, and Parameter Efficiency

The first chart compares predictive performance. The second shows whether additional parameters translate into better validation accuracy.

In [ ]:
if architecture_summary is not None:
    results = architecture_summary.sort_values('validation_accuracy_mean', ascending=False).copy()
    labels_for_plot = results['architecture'].map(architecture_labels)
    positions = np.arange(len(results))
    bar_width = 0.38
    accuracy_sd = results['validation_accuracy_std'].fillna(0)
    macro_f1_sd = results['validation_macro_f1_std'].fillna(0)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    axes[0].bar(
        positions - bar_width / 2, results['validation_accuracy_mean'], bar_width,
        yerr=accuracy_sd, capsize=4, label='Accuracy', color='steelblue',
    )
    axes[0].bar(
        positions + bar_width / 2, results['validation_macro_f1_mean'], bar_width,
        yerr=macro_f1_sd, capsize=4, label='Macro F1', color='darkorange',
    )
    axes[0].set(
        xticks=positions, xticklabels=labels_for_plot, ylim=(0, 1),
        ylabel='Validation score', title='Validation performance across seeds',
    )
    axes[0].tick_params(axis='x', rotation=20)
    axes[0].legend()

    axes[1].scatter(
        results['trainable_parameters'], results['validation_accuracy_mean'],
        s=90, color='seagreen',
    )
    for row in results.itertuples(index=False):
        axes[1].annotate(
            architecture_labels[row.architecture],
            (row.trainable_parameters, row.validation_accuracy_mean),
            xytext=(5, 5), textcoords='offset points',
        )
    axes[1].set_xscale('log')
    axes[1].set(
        xlabel='Trainable parameters (log scale)', ylabel='Mean validation accuracy',
        title='Accuracy versus model capacity', ylim=(0, 1),
    )
    plt.show()
else:
    print('Run the comparison before plotting aggregate results.')

### Learning Curves by Architecture

Each line is one seed. This reveals unstable training, early plateaus, or overfitting that a single best-epoch number can hide.

In [ ]:
history_files = sorted(COMPARISON_ROOT.glob('*/seed_*/history.csv'))
if history_files:
    fig, axis = plt.subplots(figsize=(11, 6), constrained_layout=True)
    colors = dict(zip(ARCHITECTURES_TO_COMPARE, plt.cm.tab10.colors))
    labeled_architectures = set()
    for history_path in history_files:
        architecture = history_path.parents[1].name
        seed_name = history_path.parent.name
        run_history = pd.read_csv(history_path)
        label = architecture_labels[architecture] if architecture not in labeled_architectures else None
        axis.plot(
            run_history['epoch'], run_history['val_accuracy'],
            color=colors[architecture], alpha=0.72, label=label,
        )
        labeled_architectures.add(architecture)
        best_row = run_history.loc[run_history['val_accuracy'].idxmax()]
        axis.scatter(best_row['epoch'], best_row['val_accuracy'], color=colors[architecture], s=20)
    axis.set(
        xlabel='Epoch', ylabel='Validation accuracy', ylim=(0, 1),
        title='Validation learning curves (one line per seed)',
    )
    axis.legend()
    plt.show()
else:
    print('No architecture histories found at:', COMPARISON_ROOT)

## Architecture Selection

Choose the architecture with validation evidence, considering mean accuracy, macro F1, variability, parameter count, and learning behavior. The code below suggests the highest mean-validation-accuracy architecture and its strongest seed checkpoint; it does not change `FINAL_CHECKPOINT` automatically. Review the evidence and set that path manually in the configuration cell before touching the test set.

In [ ]:
if architecture_summary is not None:
    suggested_architecture = architecture_summary.sort_values(
        'validation_accuracy_mean', ascending=False
    ).iloc[0]['architecture']
    suggested_run = (
        architecture_runs[architecture_runs['architecture'] == suggested_architecture]
        .sort_values('validation_accuracy', ascending=False)
        .iloc[0]
    )
    suggested_checkpoint = Path(suggested_run['run_dir']) / 'best.pt'
    print('Suggested architecture from validation:', suggested_architecture)
    print('Strongest validation checkpoint for final evaluation:', suggested_checkpoint)
    print('After reviewing the full evidence, set FINAL_CHECKPOINT to this path manually.')
else:
    print('Architecture selection is unavailable until the comparison is complete.')

## Held-out Test Evaluation

This section is deliberately gated. Set `FINAL_CHECKPOINT` to the validation-selected checkpoint and then set `RUN_TEST_EVALUATION = True`. Do not repeatedly evaluate alternatives on the test set, because that turns the test split into another validation set.

In [ ]:
if RUN_TEST_EVALUATION:
    checkpoint_model, checkpoint_classes, checkpoint_config, checkpoint = load_checkpoint(
        FINAL_CHECKPOINT, device
    )
    assert checkpoint_classes == classes
    assert checkpoint_config == spectrogram_config

    predictions, targets, confidences = [], [], []
    checkpoint_model.eval()
    with torch.inference_mode():
        for batch_specs, batch_labels, _ in test_loader:
            probabilities = checkpoint_model(batch_specs.to(device)).softmax(1).cpu()
            predictions.extend(probabilities.argmax(1).tolist())
            targets.extend(batch_labels.tolist())
            confidences.extend(probabilities.max(1).values.tolist())

    test_report = classification_report(
        targets, predictions, target_names=classes, output_dict=True, zero_division=0
    )
    print('Architecture:', checkpoint.get('architecture', 'crnn'))
    print('Checkpoint epoch:', checkpoint.get('epoch', 'not recorded'))
    print(f'Test accuracy: {test_report["accuracy"]:.2%}')
    print(f'Test macro F1: {test_report["macro avg"]["f1-score"]:.2%}')
    display(pd.DataFrame(test_report).T.loc[list(classes) + ['macro avg', 'weighted avg']])
else:
    print('Held-out test evaluation is disabled. Select the final model first.')

### Confusion Matrix

Rows show true species and columns show predictions. Inspect off-diagonal cells for systematic species confusions.

In [ ]:
if RUN_TEST_EVALUATION and 'test_report' in globals():
    matrix = confusion_matrix(targets, predictions, labels=range(len(classes)))
    display_matrix = ConfusionMatrixDisplay(matrix, display_labels=classes)
    fig, axis = plt.subplots(figsize=(7, 5.5), constrained_layout=True)
    display_matrix.plot(ax=axis, cmap='Blues', colorbar=False, values_format='d')
    axis.set_title('Final held-out test confusion matrix')
    plt.xticks(rotation=20, ha='right')
    plt.show()
    print(f'Mean prediction confidence: {np.mean(confidences):.2%}')
else:
    print('No final test predictions are available.')

## Takeaways

Complete this section only after the full multi-seed comparison has run. Report: (1) mean validation accuracy and macro F1 with sample standard deviation, (2) parameter counts, (3) learning-curve behavior, (4) the validation-based selection rationale, and (5) one final held-out test result.

Recorded final selection: the CRNN seed-777 checkpoint was selected from validation evidence (92.10% validation accuracy, 92.06% validation macro F1 at epoch 19) and evaluated once on the held-out test split (89.98% accuracy, 90.16% macro F1). The legacy residual checkpoint remains documented separately because earlier generator reports used it.

Classifier scores remain only one signal for generated audio. A closed-set classifier cannot determine realism or reject unknown/noisy samples, so Stage 7 should also include listening tests and spectrogram inspection.